# Egyptian Arabic Voice-Agent Evaluation

Evaluates whether an OpenAI voice model can understand and respond naturally to
**Egyptian Arabic** customer-service utterances.

**Scope of this notebook (initial pass):** test case `EG001` only. See
`_docs/specs.md` for the full evaluation plan; this notebook implements the
first, smallest possible slice of it.

## API design (read this before the code)

Audio input to OpenAI's `gpt-audio` chat-completions model does **not**
support Structured Outputs (`response_format: json_schema`) — this was
confirmed empirically against the current API (see below) and raises
`BadRequestError: 'response_format' of type 'json_schema' is not supported
with this model`. So one call cannot cleanly give us a verbatim transcription
*and* structured intent/entities *and* a natural spoken reply. Instead this
notebook makes three simple, purpose-built calls:

1. **`client.audio.transcriptions.create`** (`gpt-4o-transcribe`) — a
   dedicated speech-to-text call for a verbatim transcription. This is what
   the Word-Error-Rate style "Speech Recognition" metric in the spec (§9.1)
   should be measured against, since it isn't blended with the model
   paraphrasing/cleaning up the wording the way a generative reply would.
2. **`client.chat.completions.create`** with `model="gpt-audio"` and the
   test-case audio as `input_audio` content, `modalities=["text"]` — the
   voice agent's natural-language spoken reply, in Egyptian Arabic. (No audio
   output is requested, per the "do not generate more audio" instruction for
   this task.)
3. **`client.chat.completions.create`** with a plain text model
   (`gpt-4o-mini`) and Structured Outputs — classifies the *transcription*
   from step 1 into one of the dataset's known intents and extracts entities
   as `key=value;key=value` pairs, matching the `expected_entities` format
   already used in the CSV. This is plain NLU classification of the model's
   own transcription, not an LLM-as-judge quality evaluator.

Transcription correctness is checked automatically via simple normalized
string comparison. Intent and entity correctness are checked by comparing the
structured prediction to the CSV's `expected_intent` / `expected_entities`.
**Response quality is left for manual human rating** (1-5, per spec §10) —
no automated judge is used for that, per this task's instructions.


In [1]:
import base64
import csv
import json
import re
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio, display
from openai import OpenAI

load_dotenv()
client = OpenAI()  # reads OPENAI_API_KEY from the environment; never print the key

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CSV_PATH = REPO_ROOT / "egyptian_arabic_test_cases.csv"
AUDIO_DIR = REPO_ROOT / "audio" / "egyptian"
RESULTS_PATH = REPO_ROOT / "results" / "egyptian_results.csv"
# Generated agent-response audio is kept separate from the source test-case
# audio in AUDIO_DIR, since it's a model output, not part of the dataset.
RESPONSE_AUDIO_DIR = REPO_ROOT / "results" / "audio" / "egyptian"


## 1. Load the test case (EG001 only)

In [2]:
with open(CSV_PATH, encoding="utf-8-sig") as f:
    all_cases = list(csv.DictReader(f))

# Known intent vocabulary for this dataset, used as the classifier's enum
# below (computed from the CSV rather than hardcoded, so it stays correct if
# the dataset grows).
KNOWN_INTENTS = sorted({row["expected_intent"] for row in all_cases if row["expected_intent"]})

case = next(row for row in all_cases if row["id"] == "EG001")
audio_path = AUDIO_DIR / f"{case['id']}.mp3"

print("Test case:")
print(f"  ID:                {case['id']}")
print(f"  Dialect:            {case['dialect']}")
print(f"  Category:           {case['category']}")
print(f"  Expected text:      {case['text']}")
print(f"  Expected intent:    {case['expected_intent']}")
print(f"  Expected entities:  {case['expected_entities'] or '(none)'}")
print(f"  Audio file:         {audio_path}")
assert audio_path.exists(), f"missing audio file: {audio_path}"

display(Audio(filename=str(audio_path)))


Test case:
  ID:                EG001
  Dialect:            egyptian
  Category:           greeting
  Expected text:      السلام عليكم
  Expected intent:    greeting
  Expected entities:  (none)
  Audio file:         /workspaces/arabic-voice-agent-evaluation/audio/egyptian/EG001.mp3


## 2. Call 1 — dedicated transcription (`gpt-4o-transcribe`)

A verbatim speech-to-text transcription of the audio, used for the
speech-recognition accuracy check.

In [3]:
t0 = time.time()
with open(audio_path, "rb") as f:
    transcription_resp = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=f,
        language="ar",
        response_format="json",
    )
transcription_latency_ms = round((time.time() - t0) * 1000)

transcription = transcription_resp.text
print("Transcription:", transcription)
print("Latency (ms):", transcription_latency_ms)


Transcription: السلام عليكم
Latency (ms): 901


## 3. Call 2 — voice agent's spoken reply (`gpt-audio`)

Sends the raw audio to the audio-input chat-completions model and asks it to
reply the way a customer-service voice agent would, in Egyptian Arabic.
`modalities=["text", "audio"]` requests **both**: the model returns the reply
as synthesized speech (`message.audio.data`, base64-encoded) plus a text
transcript of that speech (`message.audio.transcript` — note `message.content`
itself is `None` once audio output is requested). The response audio is saved
next to the results CSV so it can be listened to directly here.

In [4]:
AGENT_SYSTEM_PROMPT = (
    "You are a customer-service voice agent for a clinic that speaks Egyptian "
    "Arabic. Listen to the customer audio and reply naturally in Egyptian "
    "Arabic, as a warm, concise customer-service agent would on the phone. "
    "Do not translate to English or switch to Modern Standard Arabic."
)

with open(audio_path, "rb") as f:
    audio_b64 = base64.b64encode(f.read()).decode("utf-8")

t0 = time.time()
agent_resp = client.chat.completions.create(
    model="gpt-audio",
    modalities=["text", "audio"],
    audio={"voice": "marin", "format": "mp3"},
    messages=[
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": "mp3"}}
            ],
        },
    ],
)
agent_latency_ms = round((time.time() - t0) * 1000)

agent_audio = agent_resp.choices[0].message.audio
agent_response = agent_audio.transcript

RESPONSE_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
response_audio_path = RESPONSE_AUDIO_DIR / (case["id"] + "_response.mp3")
with open(response_audio_path, "wb") as f:
    f.write(base64.b64decode(agent_audio.data))

print("Agent response:", agent_response)
print("Latency (ms):", agent_latency_ms)
print("Saved response audio to:", response_audio_path)

display(Audio(filename=str(response_audio_path)))


Agent response: وعليكم السلام! أهلاً وسهلاً بحضرتك. إزاي أقدر أساعد؟ محتاج موعد، ولا حابب تستفسر عن حاجة معينة؟
Latency (ms): 3225
Saved response audio to: /workspaces/arabic-voice-agent-evaluation/results/audio/egyptian/EG001_response.mp3


## 4. Call 3 — intent & entity classification (`gpt-4o-mini`, Structured Outputs)

Classifies the transcription from Call 1 into one of the dataset's known
intents and extracts entities in the same `key=value;key=value` format used
by `expected_entities` in the CSV, so the two can be compared directly.

In [5]:
INTENT_ENTITY_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {"type": "string", "enum": KNOWN_INTENTS},
        "entities": {
            "type": "string",
            "description": "key=value pairs separated by ';', empty string if none",
        },
    },
    "required": ["intent", "entities"],
    "additionalProperties": False,
}

CLASSIFIER_SYSTEM_PROMPT = (
    "Classify the customer utterance transcribed below (Egyptian Arabic, "
    "customer-service voice-agent context) into exactly one intent from the "
    "allowed list, and extract any relevant entities as key=value pairs "
    "separated by semicolons (keys like date,time,day,service,time_period,"
    "date_range,earliest_available,phone_number). Leave entities as an empty "
    "string if none apply."
)

t0 = time.time()
classify_resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": CLASSIFIER_SYSTEM_PROMPT},
        {"role": "user", "content": transcription},
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {"name": "intent_entities", "schema": INTENT_ENTITY_SCHEMA, "strict": True},
    },
)
classify_latency_ms = round((time.time() - t0) * 1000)

classification = json.loads(classify_resp.choices[0].message.content)
predicted_intent = classification["intent"]
predicted_entities = classification["entities"]

print("Predicted intent:  ", predicted_intent)
print("Predicted entities:", predicted_entities or "(none)")
print("Latency (ms):", classify_latency_ms)


Predicted intent:   greeting
Predicted entities: (none)
Latency (ms): 1882


## 5. Summary: test case vs. model result

In [6]:
total_latency_ms = transcription_latency_ms + agent_latency_ms + classify_latency_ms

print("=" * 60)
print("TEST CASE")
print("=" * 60)
print(f"ID:                {case['id']}")
print(f"Dialect:           {case['dialect']}")
print(f"Category:          {case['category']}")
print(f"Expected text:     {case['text']}")
print(f"Expected intent:   {case['expected_intent']}")
print(f"Expected entities: {case['expected_entities'] or '(none)'}")
print()
print("=" * 60)
print("MODEL RESULT")
print("=" * 60)
print(f"Transcription:     {transcription}")
print(f"Predicted intent:  {predicted_intent}")
print(f"Predicted entities:{predicted_entities or ' (none)'}")
print(f"Agent response:    {agent_response}")
print(f"Response audio:    {response_audio_path}")
print()
print(f"Latency - transcription: {transcription_latency_ms} ms")
print(f"Latency - agent response: {agent_latency_ms} ms")
print(f"Latency - intent/entity classification: {classify_latency_ms} ms")
print(f"Latency - total: {total_latency_ms} ms")


TEST CASE
ID:                EG001
Dialect:           egyptian
Category:          greeting
Expected text:     السلام عليكم
Expected intent:   greeting
Expected entities: (none)

MODEL RESULT
Transcription:     السلام عليكم
Predicted intent:  greeting
Predicted entities: (none)
Agent response:    وعليكم السلام! أهلاً وسهلاً بحضرتك. إزاي أقدر أساعد؟ محتاج موعد، ولا حابب تستفسر عن حاجة معينة؟
Response audio:    /workspaces/arabic-voice-agent-evaluation/results/audio/egyptian/EG001_response.mp3

Latency - transcription: 901 ms
Latency - agent response: 3225 ms
Latency - intent/entity classification: 1882 ms
Latency - total: 6008 ms


Listen to the original customer audio and the agent's spoken reply side by side:

In [7]:
print("Customer said:")
display(Audio(filename=str(audio_path)))
print("Agent replied:")
display(Audio(filename=str(response_audio_path)))


Customer said:


Agent replied:


## 6. Simple initial evaluation

Transcription correctness uses simple normalized-text comparison (strip
Arabic diacritics/punctuation, collapse whitespace) per spec §9.1 — this is
a placeholder exact-match check, not full Word Error Rate (that belongs in
`src/metrics.py` later). Intent and entity correctness compare the
structured prediction to the CSV ground truth directly.

**Response quality is a manual field below** — read `agent_response` above
and rate it yourself; no automated judge is used for this.

In [8]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]")
PUNCTUATION = re.compile(r"[\u060C\u061B\u061F.,!?;:\"'ـ]")


def normalize_arabic(text: str) -> str:
    """Very simple normalization: strip diacritics/punctuation, collapse whitespace."""
    text = ARABIC_DIACRITICS.sub("", text)
    text = PUNCTUATION.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_entities(entities: str) -> set:
    """Turn 'a=1;b=2' into {('a','1'), ('b','2')} for order-insensitive comparison."""
    pairs = [p.strip() for p in entities.split(";") if p.strip()]
    return {tuple(p.split("=", 1)) for p in pairs if "=" in p}


transcription_correct = normalize_arabic(transcription) == normalize_arabic(case["text"])
intent_correct = predicted_intent == case["expected_intent"]
entities_correct = normalize_entities(predicted_entities) == normalize_entities(case["expected_entities"])

print(f"transcription_correct: {transcription_correct}")
print(f"intent_correct:        {intent_correct}")
print(f"entities_correct:      {entities_correct}")


transcription_correct: True
intent_correct:        True
entities_correct:      True


In [9]:
# --- Manual evaluation: fill these in by hand after reading/listening above ---
# response_quality: 1 (Poor) - 5 (Excellent), per spec section 10
response_quality = 5
notes = "Understood the greeting correctly and replied naturally in Egyptian Arabic."


## 7. Save result to `results/egyptian_results.csv`

In [10]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

fieldnames = [
    "id", "dialect", "category", "expected_text", "transcription",
    "expected_intent", "predicted_intent", "expected_entities", "predicted_entities",
    "transcription_correct", "intent_correct", "entities_correct",
    "response", "response_audio_path", "latency_ms", "notes",
]

row = {
    "id": case["id"],
    "dialect": case["dialect"],
    "category": case["category"],
    "expected_text": case["text"],
    "transcription": transcription,
    "expected_intent": case["expected_intent"],
    "predicted_intent": predicted_intent,
    "expected_entities": case["expected_entities"],
    "predicted_entities": predicted_entities,
    "transcription_correct": transcription_correct,
    "intent_correct": intent_correct,
    "entities_correct": entities_correct,
    "response": agent_response,
    "response_audio_path": response_audio_path.relative_to(REPO_ROOT).as_posix(),
    "latency_ms": total_latency_ms,
    "notes": notes,
}

file_exists = RESULTS_PATH.exists()
existing_rows = []
if file_exists:
    with open(RESULTS_PATH, encoding="utf-8") as f:
        existing_rows = [r for r in csv.DictReader(f) if r["id"] != case["id"]]

with open(RESULTS_PATH, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in existing_rows:
        writer.writerow(r)
    writer.writerow(row)

print(f"Saved result for {case['id']} to {RESULTS_PATH}")


Saved result for EG001 to /workspaces/arabic-voice-agent-evaluation/results/egyptian_results.csv
